# Deep dive 2/13: `products`

More exhaustive than the `customers` notebook, per request -- `products` has more moving parts (a UNIQUE constraint, an FK to `customers` *and* one to `branches`, a currency conversion join against `silver.v_fx_latest_to_usd`, and fields that are only meaningful for a subset of `product_type` values), so there's more that can go quietly wrong without erroring.

Source of truth for what Silver *should* do: `data_pipelines/silver/table_specs.py`'s `products_spec`. Source of truth for what the data *actually* looks like: this notebook, run against your real `data/latam_bank.duckdb`. Data dictionary declares 400,000 rows, Source: Core Banking, Partition: monthly_snapshot.

Run top to bottom. Checklist near the end; free cells after that.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 1. Column inventory -- Bronze vs. Silver vs. `products_spec`

`products_spec` adds two derived columns Bronze doesn't have at all: `current_balance_usd` and `credit_limit_usd` (computed via a join to `silver.v_fx_latest_to_usd`, not a straight cast). Confirming both the straight-through columns and these two derived ones land correctly.

In [ ]:
expected_silver_columns = [
    "product_id", "customer_id", "product_type", "product_number", "currency",
    "current_balance", "current_balance_usd", "credit_limit", "credit_limit_usd",
    "interest_rate", "opening_date", "expiration_date", "opening_branch_id",
    "product_status", "opening_channel", "has_linked_app", "days_past_due",
    "last_transaction_date", "last_updated",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_products'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.dim_products:", missing or "none")
print("Extra in silver.dim_products:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

In [ ]:
print("Bronze columns:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='products' ORDER BY ordinal_position
"""))

print("Silver columns:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_products' ORDER BY ordinal_position
"""))

## 2. Row-count reconciliation -- including a fan-out check

Same dedup invariant as `customers`, but `products_spec` also does a `LEFT JOIN` to `silver.v_fx_latest_to_usd` to compute the USD columns. If that view ever has more than one row per `source_currency`, the join fans out and Silver would end up with **more** rows than Bronze has distinct `product_id` -- a bug that row-count alone can catch even without inspecting the FX logic directly (section 7 does that part).

In [ ]:
bronze_total = q("SELECT count(*) AS n FROM bronze.products")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT product_id) AS n FROM bronze.products")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.dim_products")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT product_id) AS n FROM silver.dim_products")["n"][0]

print(f"bronze.products:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct product_id")
print(f"silver.dim_products:  {silver_total:,} rows, {silver_distinct_pk:,} distinct product_id")
print()
print("Duplicate raw rows in Bronze (resends/reloads):", bronze_total - bronze_distinct_pk)
if silver_total == bronze_distinct_pk:
    print("OK -- row count matches, no FX-join fan-out")
elif silver_total > bronze_distinct_pk:
    print("FAN-OUT -- silver has MORE rows than distinct product_id, the FX join is duplicating rows")
else:
    print("MISMATCH -- silver has FEWER rows than distinct product_id, investigate")

In [ ]:
# Any duplicate product_id left in Silver itself? Should always be empty.
q("SELECT product_id, count(*) AS n FROM silver.dim_products GROUP BY 1 HAVING count(*) > 1")

In [ ]:
# If Bronze has any duplicate product_id, confirm Silver kept the one with the latest _ingested_at
# (dedup_order_by="p._ingested_at DESC" in products_spec). Empty result if Bronze has no dupes.
q("""
    WITH dupes AS (
        SELECT product_id FROM bronze.products GROUP BY 1 HAVING count(*) > 1
    ), ranked AS (
        SELECT b.product_id, b._ingested_at,
               row_number() OVER (PARTITION BY b.product_id ORDER BY b._ingested_at DESC) AS rn
        FROM bronze.products b JOIN dupes d USING (product_id)
    )
    SELECT r.product_id, r._ingested_at AS latest_bronze_ingested_at, s.last_updated
    FROM ranked r JOIN silver.dim_products s ON s.product_id = r.product_id
    WHERE r.rn = 1
    LIMIT 20
""")

## 3. NOT NULL audit

Dictionary NOT NULL columns: `product_id`, `customer_id`, `product_type`, `product_number`, `currency`, `current_balance`, `opening_date`, `opening_branch_id`, `product_status`, `opening_channel`, `has_linked_app`, `last_updated`. `credit_limit`, `interest_rate`, `expiration_date`, `days_past_due`, `last_transaction_date` are legitimately nullable (credit-specific or optional fields) -- see section 10 for whether their null rate lines up with the product mix, not just whether they're populated.

In [ ]:
declared_not_null = [
    "product_id", "customer_id", "product_type", "product_number", "currency",
    "current_balance", "opening_date", "opening_branch_id", "product_status",
    "opening_channel", "has_linked_app", "last_updated",
]

rows = []
total = q("SELECT count(*) AS n FROM bronze.products")["n"][0]
for col in declared_not_null:
    nulls = q(f"""
        SELECT count(*) AS n FROM bronze.products
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "actual_nulls": nulls,
                 "null_rate": round(nulls / total, 4) if total else None})

pd.DataFrame(rows)

## 4. UNIQUE constraint audit -- `product_number`

Declared `NOT NULL, UNIQUE` in the dictionary. Silver carries it through untouched (no transform), so any Bronze-level duplication passes straight into Silver.

In [ ]:
q("""
    SELECT
        count(*) AS total,
        count(DISTINCT product_number) AS distinct_product_number,
        count(*) FILTER (WHERE product_number IS NULL OR trim(product_number) = '') AS nulls
    FROM bronze.products
""")

In [ ]:
# Which product_number values repeat, and is it the same product_id each time (a resend) or two
# genuinely different products sharing a number (a real uniqueness violation)?
q("""
    SELECT product_number, count(*) AS n, count(DISTINCT product_id) AS distinct_product_ids
    FROM bronze.products
    GROUP BY 1 HAVING count(*) > 1
    ORDER BY n DESC
    LIMIT 20
""")

## 5. Domain checks

`product_status` is the only one of these `contracts.py` currently gates (`{Active, Closed, Blocked, Suspended}` -- matches the dictionary here, unlike `customers.customer_status`). `product_type`, `currency`, and `opening_channel` aren't gated at all -- worth seeing what's actually in them, especially since the dictionary's `product_type` cell was truncated in the PDF table and can't be trusted as a complete list.

In [ ]:
# product_type -- dictionary text was cut off mid-list in the PDF table, so treat this query,
# not the dictionary, as the authoritative list.
q("SELECT product_type, count(*) AS n FROM bronze.products GROUP BY 1 ORDER BY n DESC")

In [ ]:
# currency -- dictionary: MXN, COP, ARS, USD
q("SELECT currency, count(*) AS n FROM bronze.products GROUP BY 1 ORDER BY n DESC")

In [ ]:
# product_status -- dictionary + contracts.py both declare: Active, Blocked, Closed, Suspended
q("SELECT product_status, count(*) AS n FROM bronze.products GROUP BY 1 ORDER BY n DESC")

In [ ]:
# opening_channel -- dictionary: Branch, Web, App, Call Center
q("SELECT opening_channel, count(*) AS n FROM bronze.products GROUP BY 1 ORDER BY n DESC")

## 6. Foreign key integrity -- reproduced directly, not just trusted from the quality report

`customer_id -> customers.customer_id` and `opening_branch_id -> branches.branch_id` are both declared in `contracts.py`. The branch one was already found clean (0 orphans) during the branch_id investigation -- re-running it here directly against Silver, alongside the customer one, so this notebook doesn't have to take that on faith.

In [ ]:
q("""
    SELECT
        count(*) AS total_products,
        count(*) FILTER (WHERE customer_id IS NOT NULL) AS customer_id_populated,
        count(*) FILTER (
            WHERE customer_id IS NOT NULL
            AND NOT EXISTS (SELECT 1 FROM silver.dim_customers c WHERE c.customer_id = p.customer_id)
        ) AS customer_id_orphans
    FROM silver.dim_products p
""")

In [ ]:
q("""
    SELECT
        count(*) AS total_products,
        count(*) FILTER (WHERE opening_branch_id IS NOT NULL) AS branch_id_populated,
        count(*) FILTER (
            WHERE opening_branch_id IS NOT NULL
            AND NOT EXISTS (SELECT 1 FROM silver.dim_branches b WHERE b.branch_id = p.opening_branch_id)
        ) AS branch_id_orphans
    FROM silver.dim_products p
""")

## 7. Currency conversion correctness -- the part that isn't a simple cast

`current_balance_usd`/`credit_limit_usd` come from `TRY_CAST(...) * CASE WHEN currency='USD' THEN 1 ELSE fx.rate_to_usd END`, joined against `silver.v_fx_latest_to_usd`. Three ways this can go quietly wrong without erroring: (a) a currency in `products` that `v_fx_latest_to_usd` doesn't cover, silently producing NULL instead of a converted amount; (b) the USD passthrough not being exact; (c) a non-USD conversion using the wrong rate.

In [ ]:
# (a) FX coverage -- every currency used in products should have a row in v_fx_latest_to_usd.
print("Currencies in products:")
display(q("SELECT DISTINCT currency FROM bronze.products ORDER BY 1"))

print("Currencies covered by v_fx_latest_to_usd:")
display(q("SELECT * FROM silver.v_fx_latest_to_usd ORDER BY source_currency"))

In [ ]:
# Rows where current_balance is populated but current_balance_usd came out NULL -- the FX join
# silently failed for these. Should be 0 given full coverage above; if not, this is exactly the
# kind of gap that wouldn't show up as an error anywhere.
q("""
    SELECT currency, count(*) AS n
    FROM silver.dim_products
    WHERE current_balance IS NOT NULL AND current_balance_usd IS NULL
    GROUP BY 1
""")

In [ ]:
# (b) USD passthrough -- for currency='USD' rows, current_balance_usd should equal current_balance
# exactly (rate forced to 1), not go through a real (and irrelevant) FX rate.
q("""
    SELECT count(*) AS mismatched_usd_passthrough
    FROM silver.dim_products
    WHERE currency = 'USD'
      AND current_balance IS NOT NULL
      AND current_balance_usd IS NOT NULL
      AND abs(current_balance - current_balance_usd) > 0.01
""")

In [ ]:
# (c) Non-USD conversion -- spot-check a sample against an independent re-computation using the
# same v_fx_latest_to_usd view, to catch a join predicate or rounding bug.
q("""
    SELECT
        p.product_id, p.currency, p.current_balance, p.current_balance_usd,
        fx.rate_to_usd,
        round(p.current_balance * fx.rate_to_usd, 2) AS recomputed_usd,
        abs(p.current_balance_usd - round(p.current_balance * fx.rate_to_usd, 2)) AS diff
    FROM silver.dim_products p
    LEFT JOIN silver.v_fx_latest_to_usd fx ON fx.source_currency = p.currency
    WHERE p.currency <> 'USD' AND p.current_balance IS NOT NULL
    ORDER BY diff DESC
    LIMIT 10
""")

In [ ]:
# Same three checks, but for credit_limit_usd -- it's only populated for a subset of products
# (credit_limit itself is nullable), so NULL alone isn't a bug here; only a populated
# credit_limit with a NULL credit_limit_usd is.
q("""
    SELECT currency, count(*) AS n
    FROM silver.dim_products
    WHERE credit_limit IS NOT NULL AND credit_limit_usd IS NULL
    GROUP BY 1
""")

## 8. Numeric range sanity

In [ ]:
q("""
    SELECT
        min(current_balance) AS min_balance, max(current_balance) AS max_balance,
        count(*) FILTER (WHERE current_balance < 0) AS negative_balances,
        count(*) FILTER (WHERE current_balance IS NULL) AS nulls
    FROM silver.dim_products
""")

In [ ]:
q("""
    SELECT
        min(credit_limit) AS min_credit_limit, max(credit_limit) AS max_credit_limit,
        count(*) FILTER (WHERE credit_limit < 0) AS negative_credit_limits,
        count(*) FILTER (WHERE credit_limit IS NOT NULL) AS populated,
        count(*) AS total
    FROM silver.dim_products
""")

In [ ]:
q("""
    SELECT
        min(interest_rate) AS min_rate, max(interest_rate) AS max_rate,
        count(*) FILTER (WHERE interest_rate < 0) AS negative_rates,
        count(*) FILTER (WHERE interest_rate IS NOT NULL) AS populated
    FROM silver.dim_products
""")

In [ ]:
q("""
    SELECT
        min(days_past_due) AS min_dpd, max(days_past_due) AS max_dpd,
        count(*) FILTER (WHERE days_past_due < 0) AS negative_dpd,
        count(*) FILTER (WHERE days_past_due IS NOT NULL) AS populated,
        count(*) AS total
    FROM silver.dim_products
""")

## 9. Date sanity

In [ ]:
# opening_date vs. the dataset's overall June 2023 - June 2026 window (note: for customers this
# assumption didn't hold for registration_date -- check it fresh here rather than assume the same
# conclusion applies).
q("""
    SELECT
        min(opening_date) AS earliest_opening, max(opening_date) AS latest_opening,
        count(*) FILTER (WHERE opening_date < DATE '2023-06-17') AS before_window,
        count(*) FILTER (WHERE opening_date > DATE '2026-06-17') AS after_window
    FROM silver.dim_products
""")

In [ ]:
# expiration_date should be after opening_date when populated (it's "for term products" per the
# dictionary, so NULL is expected for most rows -- only checking internal consistency where set).
q("""
    SELECT
        count(*) FILTER (WHERE expiration_date IS NOT NULL) AS populated,
        count(*) FILTER (WHERE expiration_date IS NOT NULL AND expiration_date <= opening_date) AS expires_before_or_on_opening
    FROM silver.dim_products
""")

In [ ]:
# last_transaction_date should never precede opening_date, and last_updated should never precede
# either. Also checking for the same kind of "future" anomaly customers.last_updated had.
q("""
    SELECT
        max(last_updated) AS latest_last_updated,
        count(*) FILTER (WHERE last_transaction_date IS NOT NULL AND TRY_CAST(last_transaction_date AS DATE) < opening_date) AS txn_before_opening,
        count(*) FILTER (WHERE last_updated < opening_date) AS updated_before_opening,
        count(*) FILTER (WHERE last_updated > TIMESTAMP '2026-06-17 23:59:59') AS updated_after_dataset_window
    FROM silver.dim_products
""")

## 10. Cross-field consistency -- are credit-only fields actually credit-only?

`credit_limit` and `days_past_due` are documented as credit-product fields. This checks whether that's actually how the data is shaped: populated mostly for credit-type products, NULL for deposit-type ones (or not -- worth knowing either way, since nothing in Silver enforces this).

In [ ]:
q("""
    SELECT
        product_type,
        count(*) AS n,
        count(*) FILTER (WHERE credit_limit IS NOT NULL) AS has_credit_limit,
        round(100.0 * count(*) FILTER (WHERE credit_limit IS NOT NULL) / count(*), 1) AS pct_has_credit_limit,
        count(*) FILTER (WHERE days_past_due IS NOT NULL) AS has_days_past_due,
        round(100.0 * count(*) FILTER (WHERE days_past_due IS NOT NULL) / count(*), 1) AS pct_has_dpd
    FROM silver.dim_products
    GROUP BY 1
    ORDER BY n DESC
""")

## 11. `has_linked_app` -- boolean parse check, same pattern as `customers.accepts_marketing`

In [ ]:
print("Bronze raw values:")
display(q("SELECT typeof(has_linked_app), has_linked_app, count(*) AS n FROM bronze.products GROUP BY ALL ORDER BY n DESC"))

print("Silver parsed values:")
display(q("SELECT typeof(has_linked_app), has_linked_app, count(*) AS n FROM silver.dim_products GROUP BY ALL ORDER BY n DESC"))

## 12. Known issues recap

- **`opening_branch_id -> branches.branch_id`**: confirmed clean (0 orphans, 100% populated) in the earlier branch_id investigation -- section 6 re-derives this directly against your current data.
- No `customer_status`-style domain drift found here: `product_status`'s real values (`Active, Blocked, Closed, Suspended`) match both the dictionary and `contracts.py` exactly.

## Checklist

- [ ] Column inventory (1): spec/table agree, `current_balance_usd`/`credit_limit_usd` present with correct types
- [ ] Row reconciliation (2): no FX-join fan-out, no duplicate `product_id` in Silver
- [ ] NOT NULL audit (3): all 12 declared columns actually populated
- [ ] UNIQUE audit (4): `product_number` genuinely unique, or note any real collisions found
- [ ] Domain checks (5): note any `product_type`/`currency`/`opening_channel` value that looks like an error rather than a real category
- [ ] FK integrity (6): both `customer_id` and `opening_branch_id` at 0 orphans
- [ ] FX conversion (7): full currency coverage, exact USD passthrough, correct non-USD math
- [ ] Numeric ranges (8): no negative balances/limits/rates/DPD
- [ ] Date sanity (9): no expirations before opening, no transactions/updates before opening, no anomalous window violations beyond what's explained
- [ ] Cross-field consistency (10): credit_limit/days_past_due population lines up with product_type as expected, or note if it doesn't
- [ ] `has_linked_app` (11): NULL rate (if any) explained by genuinely unparseable source values

In [ ]:
con.close()
print("OK")